# LORENZ-96

In [ ]:
# LIBRARIES
from runner import run_data_assimilation
from models.lorenz96 import Lorenz96
from observation_operators import SparseGaussian
from filters import KernelDensityEstimation, VE, VP, odeint_sampler
from filters import EnsembleKalmanFilter

from utils.plots import plot_rmses, plot_timeplots, plot_particles, plot_sigmas

import os
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import ipywidgets # NEEDED TO MAKE interactive plots
import json
import itertools
from scipy.optimize import root_scalar


import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display


RANDOM_SD = 10
np.random.seed(RANDOM_SD) # TODO: remove

data_path = "inp/initial/"
# exp_path = "exp" # for data normalization
T_spinup = 2000 # NOTE now used in model parameters

In [2]:
# Define all ensemble sizes to test
ensemble_sizes = [10, 20, 40, 60, 100, 200, 400, 600]
# ensemble_sizes = [600]

sigma_min_xs = [0.1, 0.2, 0.3, 0.4]
sigma_ys = [np.sqrt(0.5)]

# Filters to run (each as a dictionary)
filter_variants = [
    {"type": "KDE", "scheduler": "VE", "sigma_max": 50},
    {"type": "KDE", "scheduler": "VP", "sigma_max": 10}
]

exp_path = "exp"
models = {}

In [3]:
# SET UP observation operator
obs_op_params = {
    'mean': 0,
    'std': np.sqrt(0.5),
    'random_sd': RANDOM_SD
}
observation_op = SparseGaussian(**obs_op_params)

# Model parameters that will be the same for every variation
global_model_params = {
    'F': 8,
    'dim': 40,
    'for_dt': 0.01, # Process timestep
    'obs_dt': 0.4, # Observation timestep
    # 'T_spinup':2000,
    'T_burnin': 2000,
    'T_steps': 4000, # Number of assimilation steps
    # 'process_noise': 10e-2, # std PROCESS
    'random_sd': RANDOM_SD,
    'initial_time': 0, 
    'obs_op': observation_op # std # REFERENCE 
}

# Filter parameters that are required for both KDE and EnKF filters
global_filter_params = {
    'obs_op': observation_op, 
    'random_sd': RANDOM_SD
} 

for ensemble_size in ensemble_sizes:

    # SET UP model * * * * * * * * * * * * * * * * * * * * * * * * * *
    dynamic_model_params = {
        'ensemble_size': ensemble_size
        # 'ref_initial_state': np.array(data['xt'][0][0].T)[T_spinup-1] # REF
    }

    model_params = {**global_model_params, **dynamic_model_params} # Merge dictionaries

    # Iterate through filter variants * * * * * * * * * * * * * * * * *
    for filter_config in filter_variants:
        filter_type = filter_config["type"]

        if filter_type == "KDE":
            scheduler = filter_config["scheduler"]
            sigma_max = filter_config["sigma_max"]

            # For saving models into dict
            kde_key = f"KDE {scheduler}"
            if kde_key not in models:
                models[kde_key] = {}  # Create empty sub-dictionary

            # Iterate through different sigma values
            for sigma_min_x in sigma_min_xs:
                for sigma_y in sigma_ys:

                    model = Lorenz96(**model_params) # Initialize model

                    # Set Up filter parameters
                    kde_filter_params = {
                        'scheduler': scheduler,
                        'sigma_min': sigma_min_x,
                        'sigma_max': sigma_max,  # Different for VP and VE
                        'sigma_y': sigma_y,
                        'N_tsteps': 1000  # Pseudo-time steps
                    }

                    filter_params = {**global_filter_params, **kde_filter_params}
                    filter = KernelDensityEstimation(**filter_params) # Initialize filter
                    
                    # Construct filename
                    filename = f"{str(model)}_M{ensemble_size}_KDE_{scheduler}_SX{sigma_min_x}_SY{sigma_y}.npz"

                    # Run model
                    model = run_data_assimilation(exp_path, filename, model, filter)

                    # Store in dictionary
                    models[kde_key][(ensemble_size, sigma_min_x, sigma_y)] = model


Loading existing data from exp/L96_M10_KDE_VE_SX0.1_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M10_KDE_VE_SX0.2_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M10_KDE_VE_SX0.3_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M10_KDE_VE_SX0.4_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M10_KDE_VP_SX0.1_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M10_KDE_VP_SX0.2_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M10_KDE_VP_SX0.3_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M10_KDE_VP_SX0.4_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M20_KDE_VE_SX0.1_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M20_KDE_VE_SX0.2_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M20_KDE_VE_SX0.3_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M20_KDE_VE_SX0.4_SY0.7071067811865476.npz.
Loading existing data from exp/L96_M20_KDE_VP_SX0.1_SY0.70710678

In [4]:
plot_rmses(models)

interactive(children=(Dropdown(description='σ_x', options=(np.float64(0.1), np.float64(0.2), np.float64(0.3), …